# SD-MAC — Schema-Validation Demo

**Component:** SD-MAC (Sector-Wide Deployable Modular Analytics Commons)

This notebook validates sample records against the SD-MAC schema registry via `shared.utilities.schema_loader`. It shows a **passing** record and a **deliberately invalid** record being caught, demonstrating the schema-conformance check.

> **Synthetic data only.** Every dataset used in this notebook is *synthetic illustrative data generated for demonstration*. It is **NOT** real agency data and is **NOT** derived from any proprietary or employer source. The notebook performs no network access and uses only public-style, open-source components.

In [1]:
import sys
from pathlib import Path

# Put the oefaf-platform repository root on sys.path so the platform
# component packages (gea, cricat, sdmac, shared) import cleanly no matter
# what working directory this notebook is executed from.
_REPO_ROOT = Path.cwd()
while not (_REPO_ROOT / 'sdmac' / 'schema_registry').is_dir():
    if _REPO_ROOT == _REPO_ROOT.parent:
        raise RuntimeError('Could not locate the oefaf-platform repo root.')
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
print('repo root located:', _REPO_ROOT.name)

repo root located: oefaf-platform


## 1. List the registry schemas and render one as JSON Schema

`load_schema` reads the registry YAML; `to_json_schema` converts it to a Draft-7 JSON Schema document used for validation.

In [2]:
import json

from shared.utilities.schema_loader import (
    load_schema,
    schema_registry_dir,
    to_json_schema,
    validate_record,
)

schemas = sorted(p.stem for p in schema_registry_dir().glob('*.yaml'))
print('registry schemas:', schemas)

reg = load_schema('supply_disruption_event')
js = to_json_schema(reg)
print('\nsupply_disruption_event -> JSON Schema (Draft-7):')
print(json.dumps(js, indent=2))

registry schemas: ['capacity_allocation_scenario', 'load_forecast_record', 'module_manifest', 'supply_disruption_event']

supply_disruption_event -> JSON Schema (Draft-7):
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "supply_disruption_event",
  "type": "object",
  "properties": {
    "event_id": {
      "type": "string",
      "description": "Stable unique identifier for the disruption event"
    },
    "detected_at_utc": {
      "type": "string",
      "format": "date-time",
      "description": "Time the event was first detected from public sources"
    },
    "commodity": {
      "enum": [
        "crude_oil",
        "natural_gas",
        "refined_products",
        "electricity",
        "lng",
        "coal",
        "agricultural"
      ],
      "description": "Affected commodity category"
    },
    "region_iso": {
      "type": "string",
      "description": "ISO 3166-1 alpha-3 country code or recognized region code (e.g., \"USA\", \"EUR\")"
    },
  

## 2. A passing record

A well-formed synthetic `supply_disruption_event` validates cleanly.

In [3]:
valid_event = {
    'event_id': 'GEA-EVT-DEMO-0001',
    'detected_at_utc': '2026-03-15T12:00:00Z',
    'commodity': 'natural_gas',
    'region_iso': 'USA',
    'source_categories': ['sanctions', 'weather'],
    'severity_score': 0.62,
    'confidence': 0.74,
    'public_evidence_refs': [
        'https://example.org/synthetic-evidence/natural_gas/USA/0001/1',
    ],
}
print('validating a well-formed synthetic record ...')
ok = validate_record(valid_event, 'supply_disruption_event')
print('PASS — record is schema-conformant:', ok)

validating a well-formed synthetic record ...
PASS — record is schema-conformant: True


## 3. A deliberately invalid record (caught)

This record has `severity_score = 1.8` (outside the registry's `0.0_to_1.0` bound) and a `commodity` value not in the enum. `validate_record` raises `jsonschema.ValidationError`, which we catch and report — the failure is never silently swallowed.

In [4]:
import jsonschema

invalid_event = {
    'event_id': 'GEA-EVT-DEMO-BAD',
    'detected_at_utc': '2026-03-15T12:00:00Z',
    'commodity': 'unobtanium',          # not in the commodity enum
    'region_iso': 'USA',
    'source_categories': ['weather'],
    'severity_score': 1.8,               # outside [0, 1]
    'confidence': 0.74,
    'public_evidence_refs': ['https://example.org/synthetic-evidence/bad/1'],
}

try:
    validate_record(invalid_event, 'supply_disruption_event')
    raise AssertionError('expected the invalid record to be rejected')
except jsonschema.ValidationError as exc:
    print('CAUGHT — invalid record correctly rejected.')
    print('failing path :', list(exc.absolute_path))
    print('message      :', exc.message)

CAUGHT — invalid record correctly rejected.
failing path : ['severity_score']
message      : 1.8 is greater than the maximum of 1


## 4. Conformance sweep over the bundled synthetic fixtures

Every bundled synthetic record validates against its registry schema, giving a 100% conformance rate on the demonstration fixtures.

In [5]:
from shared.utilities.io import read_json, repo_root

fx = repo_root() / 'shared' / 'data_sources' / 'fixtures'
checks = [
    (fx / 'gea' / 'supply_disruption_events.json', 'supply_disruption_event'),
    (fx / 'cricat' / 'load_forecast_records.json', 'load_forecast_record'),
    (fx / 'cricat' / 'capacity_allocation_scenarios.json', 'capacity_allocation_scenario'),
    (fx / 'sdmac' / 'module_manifests.json', 'module_manifest'),
]

total = 0
for path, schema_name in checks:
    records = read_json(path)
    for rec in records:
        validate_record(rec, schema_name)
    total += len(records)
    print(f'{schema_name:<32} {len(records):>2} record(s)  -> all valid')
print(f'\nconformance rate: {total}/{total} synthetic fixture records valid (100%).')

supply_disruption_event          12 record(s)  -> all valid
load_forecast_record              4 record(s)  -> all valid
capacity_allocation_scenario      3 record(s)  -> all valid
module_manifest                   4 record(s)  -> all valid

conformance rate: 23/23 synthetic fixture records valid (100%).


---

**Recap.** The registry YAML was rendered to JSON Schema, a well-formed synthetic record passed, a deliberately invalid record was caught and reported, and every bundled synthetic fixture record validated against its schema. All data is synthetic illustrative data; no proprietary or employer content is used.